# Cluster-GCN on Reddit Graph

Node Classification on Reddit: Training large-scale GCN on Reddit by partitioning nodes into subgraphs. This notebook implements the approach with `SAGEConv` inside a `K3RedditNet` model, trained with the Adam optimizer for 10 epochs, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `SAGEConv` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install git+http://github.com/anas-rz/k3-node/@main

# ==============================================================================
# K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops
import numpy as np

import k3_node
from k3_node import layers as k3_layers
from k3_node.datasets import Reddit
from k3_node.loader import ClusterData, ClusterLoader

title = "Cluster-GCN on Reddit Graph"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Dataset & Cluster-GCN Partitioning
dataset = Reddit(root="./data/Reddit")
data = dataset[0]
num_features = dataset.num_features
num_classes = dataset.num_classes

cluster_data = ClusterData(
    data,
    num_parts=1500,
    save_dir=dataset.processed_dir,
)
train_loader = ClusterLoader(cluster_data, batch_size=20, shuffle=True)

# 2. SAGEConv Model
class K3RedditNet(keras.Model):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = k3_layers.SAGEConv(in_channels, hidden_channels)
        self.conv2 = k3_layers.SAGEConv(hidden_channels, out_channels)

    def build(self, input_shape=None):
        self.conv1.build((None, self.conv1.in_channels))
        self.conv2.build((None, self.conv2.in_channels))
        self.built = True

    def call(self, inputs, edge_index=None):
        if isinstance(inputs, (tuple, list)):
            x, edge_index = inputs[0], inputs[1]
        else:
            x = inputs
        x = ops.relu(self.conv1(x, edge_index))
        return self.conv2(x, edge_index)

k3_model = K3RedditNet(num_features, 128, num_classes)

# 3. Model Compilation
k3_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.005),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
)

# 4. Multi-epoch Cluster Generator
def to_np(t, dtype=None):
    if hasattr(t, "cpu"):
        t = t.cpu()
    if hasattr(t, "detach"):
        t = t.detach()
    if hasattr(t, "numpy") and callable(t.numpy):
        t = t.numpy()
    return np.asarray(t, dtype=dtype)

def make_generator(loader):
    while True:
        for batch in loader:
            x = to_np(batch.x, dtype=np.float32)
            edge_index = to_np(batch.edge_index, dtype=np.int64)
            y = to_np(batch.y, dtype=np.int64)
            mask = to_np(batch.train_mask, dtype=np.float32) if hasattr(batch, "train_mask") else None
            if mask is not None and mask.sum() > 0:
                yield (x, edge_index), y, mask
            elif mask is None:
                yield (x, edge_index), y

print(f"Training K3-Node Cluster-GCN on {backend} backend...")
history = k3_model.fit(
    make_generator(train_loader),
    steps_per_epoch=len(train_loader),
    epochs=10,
    verbose=1,
)

print("\n✓ K3-Node execution completed successfully!")